# SemanticSCVI (geometric) on CTCL malignant cells

Trains the fork's `SemanticSCVI` geometric-loss factor model — the same **semantic_geom**
model used in `four_way_benchmark_haniffa_cd8.ipynb` (Geneformer prior, identical params) —
on the **tumor cells** of the CTCL atlas, then prints:

1. the **top loading genes per projection** (latent factor), and
2. a **hierarchical clustering** of the top-loaded genes.

Cells: `cell_type == "tumor_cell"` from the complete atlas. Batch key: `donor`.
Run on the `neural_nmf_env` GPU kernel.

In [ ]:
# ============================================================
# Parameters — edit here. Same knobs/names as four_way_benchmark_haniffa_cd8.ipynb.
# ============================================================
import hashlib
import json
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path(__file__).parent.resolve() if "__file__" in globals() else Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data starting from {start}")


def _semantic_cache_slug(kwargs, max_epochs, warmup_epochs, n_epochs_kl_warmup, hvg_top_n, n=10):
    """Stable 10-char hash of every param that affects the trained SemanticSCVI model.
    Change any of these and the cache_dir flips -> fresh train. Same params -> cache hit."""
    blob = json.dumps(
        {
            "kwargs": dict(sorted(kwargs.items())),
            "max_epochs": max_epochs,
            "warmup_epochs": warmup_epochs,
            "n_epochs_kl_warmup": n_epochs_kl_warmup,
            "hvg_top_n": hvg_top_n,
        },
        default=str, sort_keys=True,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


NB_DIR = _resolve_nb_dir()
print(f"NB_DIR = {NB_DIR}")

# ---- paths ----
ATLAS_PATH     = NB_DIR / "data" / "cache" / "mrvi_ctcl_cache.h5ad"           # complete atlas
GENE_ID_SOURCE = NB_DIR / "data" / "cache" / "cnmf_malignant_counts.h5ad"     # gene_name -> gene_id only
SEMANTIC_CACHE_GENEFORMER = NB_DIR / "data" / "mf_tumor_geneformer.pt"
MODEL_CACHE_DIR = NB_DIR / "models" / ".model_cache_semantic_geom_tumor"
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
CELL_TYPE = "tumor_cell"

# ---- Preprocessing (ref values) ----
HVG_TOP_N  = 2500
HVG_FLAVOR = "seurat_v3"
SUBSAMPLE_N = 40000        # ref value; set None to use all ~102k tumor cells

# ---- Shared model size (ref) ----
N_LATENT = 10

# ---- Batch effect ----
BATCH_KEY = "donor"        # ref used "Site"; donor is this atlas's batch

# ---- SemanticSCVI (Geneformer prior) — EXACT ref values ----
SEMANTIC_GENEFORMER_MAX_EPOCHS = 200
SEMANTIC_GENEFORMER_WARMUP_EPOCHS = 20
SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP = 100
SEMANTIC_GENEFORMER_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)
SEMANTIC_GENEFORMER_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi" / _semantic_cache_slug(
    {**SEMANTIC_GENEFORMER_KWARGS, "batch_key": BATCH_KEY},
    SEMANTIC_GENEFORMER_MAX_EPOCHS,
    SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
print(f"semantic_geom cache_dir: {SEMANTIC_GENEFORMER_CACHE_DIR}")
FORCE_TRAIN_SEMANTIC_GENEFORMER = False

# ---- interpretation (ref values) ----
PER_PROJECTION_N_TOP = 30   # top loading genes printed per projection
CLUSTER_N_TOP        = 500  # genes entering hierarchical clustering
CLUSTER_MAX_K        = 20   # silhouette search range for #clusters
TOP_PER_CLUSTER      = 25   # genes printed per hierarchical cluster

In [ ]:
import sys
if str(NB_DIR.parent) not in sys.path:
    sys.path.insert(0, str(NB_DIR.parent))   # notebooks/ holds benchmark_helpers.py

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import scvi

from benchmark_helpers import get_or_build_geneformer_map, train_or_load_semantic_scvi

scvi.settings.seed = 0
print("scvi", scvi.__version__, "| scanpy", sc.__version__)
print("CUDA available:", torch.cuda.is_available())

## Load tumor cells, attach Ensembl `gene_id`

In [ ]:
adata = sc.read_h5ad(ATLAS_PATH)
adata = adata[adata.obs["cell_type"].astype(str) == CELL_TYPE].copy()
adata.X = adata.layers["raw_counts"].copy()        # NB likelihood needs raw counts

# Geneformer tokens are keyed by Ensembl id; this atlas's var has only symbols.
src = sc.read_h5ad(GENE_ID_SOURCE, backed="r")
sym2ens = dict(zip(src.var["gene_name"].astype(str), src.var["gene_id"].astype(str)))
adata.var["gene_id"] = [sym2ens.get(s, s) for s in adata.var_names.astype(str)]
n_mapped = int(sum(g.startswith("ENSG") for g in adata.var["gene_id"]))
print(adata.shape, "| cell_type:", adata.obs["cell_type"].unique().tolist())
print(f"gene_id mapped to Ensembl: {n_mapped}/{adata.n_vars}")

In [ ]:
adata = sc.read_h5ad(ATLAS_PATH)
adata.obs.value_counts()

In [ ]:
adata.obs.cell_type.value_counts()

## Build Geneformer semantic map, then HVG-subset

In [ ]:
# Build the FULL-gene Geneformer map first (downloads Geneformer ~415 MB on first call),
# then HVG-subset adata + map together. Delete mf_tumor_geneformer.pt to force a rebuild.
semantic_map = get_or_build_geneformer_map(
    adata, SEMANTIC_CACHE_GENEFORMER, var_id_key="gene_id",
)
print("semantic_map (raw):", tuple(semantic_map.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False)
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
print("after HVG:", adata.shape, "| semantic_map:", tuple(semantic_map.shape))

In [ ]:
if SUBSAMPLE_N is not None and adata.n_obs > SUBSAMPLE_N:
    sc.pp.subsample(adata, n_obs=SUBSAMPLE_N, random_state=42)
    print(f"subsampled to {adata.n_obs} cells")
else:
    print(f"no subsampling (n_obs={adata.n_obs}, SUBSAMPLE_N={SUBSAMPLE_N})")

## Train SemanticSCVI (geometric, Geneformer prior) — GPU

In [ ]:
model = train_or_load_semantic_scvi(
    adata,
    semantic_map,
    cache_dir=SEMANTIC_GENEFORMER_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEFORMER,
    max_epochs=SEMANTIC_GENEFORMER_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    batch_key=BATCH_KEY,
    **SEMANTIC_GENEFORMER_KWARGS,
)
print("trained:", model.is_trained)

## UMAP of the SemanticSCVI embedding

In [ ]:
# UMAP of the SemanticSCVI latent embedding (batch_key=donor).
# Build a standalone AnnData from the model's registered adata so the latent and obs
# stay aligned regardless of the state of `adata` in the kernel.
z = np.asarray(model.get_latent_representation())
ad_emb = sc.AnnData(z, obs=model.adata.obs.copy())
sc.pp.neighbors(ad_emb, use_rep="X", random_state=0)
sc.tl.umap(ad_emb, random_state=0)

color = [c for c in ["stage", "study", "tissue", "donor"] if c in ad_emb.obs]
sc.pl.umap(ad_emb, color=color, ncols=2, wspace=0.3, frameon=False)

## Gene loadings (projections)

In [ ]:
W = model.get_loadings().reindex(adata.var_names)   # genes x N_LATENT, indexed by gene symbol
adata.varm["W_semantic_geom"] = W.values
adata.uns["W_semantic_geom_columns"] = list(W.columns)
print("W (loadings):", W.shape, "| columns:", list(W.columns))
W.head()

### Top loading genes per projection

In [ ]:
top_per_factor = {
    f"proj_{col}": W[col].nlargest(PER_PROJECTION_N_TOP).index.tolist()
    for col in W.columns
}
top_df = pd.DataFrame(top_per_factor)
top_df.index = [f"top_{i + 1}" for i in range(PER_PROJECTION_N_TOP)]
with pd.option_context("display.max_columns", None, "display.width", 250,
                       "display.max_colwidth", 30):
    print(top_df)
top_df

## Hierarchical clustering of top-loaded genes

Mirrors `benchmarking._get_hierarchical_clusters`: top genes by max |loading| →
correlation-distance → average linkage → #clusters chosen by silhouette.

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

max_vals = W.abs().max(axis=1)
top_idx = max_vals.sort_values(ascending=False).head(CLUSTER_N_TOP).index
subset = W.loc[top_idx]

# correlation distance is undefined for zero-variance rows; drop them
keep = subset.values.std(axis=1) > 0
if not keep.all():
    print(f"dropping {(~keep).sum()} zero-variance genes before correlation pdist")
subset = subset.loc[keep]
top_idx = subset.index

dists = pdist(subset.values, metric="correlation")
dist_matrix = squareform(dists)
Z = linkage(dists, method="average")

best_k, best_score = 2, -1.0
for k in range(2, CLUSTER_MAX_K + 1):
    labels = fcluster(Z, t=k, criterion="maxclust")
    try:
        s = silhouette_score(dist_matrix, labels, metric="precomputed")
    except ValueError:
        continue
    if s > best_score:
        best_score, best_k = s, k
final_labels = fcluster(Z, t=best_k, criterion="maxclust")
print(f"hierarchical clusters: k={best_k} (silhouette={best_score:.3f}) over {len(top_idx)} genes")

clust = pd.DataFrame(
    {"Cluster": final_labels, "max_loading": max_vals.loc[top_idx].values},
    index=top_idx,
)
for c in sorted(clust["Cluster"].unique()):
    members = clust[clust["Cluster"] == c].sort_values("max_loading", ascending=False)
    genes = members.head(TOP_PER_CLUSTER).index.tolist()
    print(f"\n=== cluster {c}  (n={len(members)}) ===")
    print(", ".join(genes))

In [ ]:
import seaborn as sns

corr = np.corrcoef(subset.values)   # gene x gene correlation across projections
g = sns.clustermap(
    pd.DataFrame(corr, index=top_idx, columns=top_idx),
    row_linkage=Z, col_linkage=Z, cmap="vlag", center=0,
    xticklabels=False, yticklabels=False, figsize=(8, 8),
)
g.fig.suptitle(
    f"semantic_geom tumor: top-{len(top_idx)} gene loading correlation (k={best_k})", y=1.01
)
out = FIG_DIR / "semantic_geom_tumor_gene_hierarchy.png"
g.savefig(out, dpi=200, bbox_inches="tight")
print("saved", out)